# Notebook 00 — Data Preparation for IRT Pipeline

**Goal**: Convert raw xlsx to CSV, split the merged cleaned dataset by collection wave into two independent samples, validate cleaning, and export sample-level datasets for all downstream IRT analysis.

- **Sample 1 (Pretest)**: `pretest_EN` — 1,035 participants
- **Sample 2 (Validation)**: `ART_prestest_responses` — ~800 participants

All IRT/dimensionality analyses use **real-author items only**; foils are retained for CTT false-alarm scoring.

In [1]:
import os, sys
import pandas as pd
import numpy as np

# Resolve project root
_MARKER = "data/processed/art_cleaned/ART_pretest_merged_EN_cleaned.csv"
PROJECT_ROOT = os.path.abspath(os.getcwd())
while PROJECT_ROOT:
    if os.path.isfile(os.path.join(PROJECT_ROOT, _MARKER)):
        break
    _parent = os.path.dirname(PROJECT_ROOT)
    if _parent == PROJECT_ROOT:
        PROJECT_ROOT = os.path.abspath(os.getcwd())
        break
    PROJECT_ROOT = _parent

CLEANED_CSV  = os.path.join(PROJECT_ROOT, _MARKER)
ITEM_META    = os.path.join(PROJECT_ROOT, "data/processed/art_cleaned/item_metadata.csv")
XLSX_PATH    = os.path.join(PROJECT_ROOT, "archive/pretest_dataset_ART_only_EN.xlsx")
OUT_DIR      = os.path.join(PROJECT_ROOT, "data/processed/irt_pipeline")
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Cleaned CSV  : {os.path.exists(CLEANED_CSV)}")
print(f"Item metadata: {os.path.exists(ITEM_META)}")
print(f"XLSX exists  : {os.path.exists(XLSX_PATH)}")
print(f"Output dir   : {OUT_DIR}")

Project root : /home/polina/Documents/Cursor_Projects/Russian Author Recognition Test Cursor
Cleaned CSV  : True
Item metadata: True
XLSX exists  : True
Output dir   : /home/polina/Documents/Cursor_Projects/Russian Author Recognition Test Cursor/data/processed/irt_pipeline


## 1. Convert XLSX to CSV (reference copy)

In [2]:
xlsx_csv_path = os.path.join(OUT_DIR, "pretest_dataset_ART_only_EN.csv")

if os.path.exists(XLSX_PATH):
    df_xlsx = pd.read_excel(XLSX_PATH, header=None)
    df_xlsx.to_csv(xlsx_csv_path, index=False, header=False)
    print(f"Converted XLSX -> CSV: {df_xlsx.shape}")
    print(f"Saved to: {xlsx_csv_path}")
else:
    print("XLSX file not found at", XLSX_PATH)
    print("Will proceed using merged cleaned CSV only.")

Converted XLSX -> CSV: (1037, 219)
Saved to: /home/polina/Documents/Cursor_Projects/Russian Author Recognition Test Cursor/data/processed/irt_pipeline/pretest_dataset_ART_only_EN.csv


## 2. Load merged cleaned dataset and item metadata

In [3]:
# The cleaned CSV has a two-row header: row 0 = labels, row 1 = codes
raw = pd.read_csv(CLEANED_CSV)
labels_row = raw.columns.tolist()
codes_row  = raw.iloc[0].tolist()

# Data starts at row index 1 (after the code row)
df = raw.iloc[1:].reset_index(drop=True).copy()
df.columns = labels_row

meta = pd.read_csv(ITEM_META)

print(f"Dataset shape (participants x columns): {df.shape}")
print(f"Item metadata entries: {len(meta)}")
print(f"Source column values:\n{df['source'].value_counts()}")

Dataset shape (participants x columns): (1835, 210)
Item metadata entries: 204
Source column values:
source
pretest_EN                1035
ART_prestest_responses     800
Name: count, dtype: int64


## 3. Identify item columns

In [4]:
DEMO_COLS = ['Submited', 'age', 'sex ', 'humanities or not', 'education and profession']
SOURCE_COL = 'source'

# Item columns = everything except demo and source
item_cols = [c for c in df.columns if c not in DEMO_COLS + [SOURCE_COL]]
print(f"Total item columns in cleaned data: {len(item_cols)}")

# Real authors vs foils from metadata
real_author_labels = meta.loc[meta['is_real_author'] == True, 'item_label'].tolist()
foil_labels        = meta.loc[meta['is_foil'] == True,        'item_label'].tolist()

# Match to column names
author_cols = [c for c in item_cols if c in real_author_labels]
foil_cols   = [c for c in item_cols if c in foil_labels]
unmatched   = [c for c in item_cols if c not in real_author_labels and c not in foil_labels]

print(f"Real author columns: {len(author_cols)}")
print(f"Foil columns:        {len(foil_cols)}")
if unmatched:
    print(f"WARNING — unmatched columns: {unmatched}")
else:
    print("All item columns matched to metadata.")

Total item columns in cleaned data: 204
Real author columns: 101
Foil columns:        103
All item columns matched to metadata.


## 4. Coerce item responses to numeric and validate binary coding

In [5]:
# Coerce all item columns to numeric (invalid -> NaN)
for col in item_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Validate: all non-NaN values should be 0 or 1
all_vals = df[item_cols].values.flatten()
non_nan  = all_vals[~np.isnan(all_vals)]
unique_vals = np.unique(non_nan)

print(f"Unique non-NaN values in item matrix: {unique_vals}")
assert set(unique_vals).issubset({0.0, 1.0}), f"Non-binary values found: {unique_vals}"
print("Binary validation PASSED — all values are 0 or 1.")

# Coerce demographics
df['age'] = pd.to_numeric(df['age'], errors='coerce')

# Overall missingness
total_cells = df[item_cols].size
missing_cells = df[item_cols].isna().sum().sum()
print(f"\nOverall missingness: {missing_cells:,} / {total_cells:,} ({100*missing_cells/total_cells:.2f}%)")

Unique non-NaN values in item matrix: [0. 1.]
Binary validation PASSED — all values are 0 or 1.

Overall missingness: 1,170 / 374,340 (0.31%)


## 5. Split by source into two samples

In [6]:
sample1 = df[df[SOURCE_COL] == 'pretest_EN'].copy().reset_index(drop=True)
sample2 = df[df[SOURCE_COL] == 'ART_prestest_responses'].copy().reset_index(drop=True)

print(f"Sample 1 (Pretest):    N = {len(sample1)}")
print(f"Sample 2 (Validation): N = {len(sample2)}")
print(f"Combined:              N = {len(sample1) + len(sample2)}")
assert len(sample1) + len(sample2) == len(df), "Split counts don't match total!"

Sample 1 (Pretest):    N = 1035
Sample 2 (Validation): N = 800
Combined:              N = 1835


## 6. Per-sample validation checks

In [7]:
def validate_sample(sample_df, name, author_cols, foil_cols):
    """Run validation checks on a sample."""
    print(f"\n{'='*60}")
    print(f"Validation: {name}  (N = {len(sample_df)})")
    print(f"{'='*60}")
    
    # Check item columns are present
    missing_author = [c for c in author_cols if c not in sample_df.columns]
    missing_foil   = [c for c in foil_cols   if c not in sample_df.columns]
    print(f"Missing author columns: {len(missing_author)}")
    print(f"Missing foil columns:   {len(missing_foil)}")
    
    # Missingness per item
    author_miss = sample_df[author_cols].isna().mean() * 100
    foil_miss   = sample_df[foil_cols].isna().mean() * 100
    
    print(f"\nAuthor items missingness:")
    print(f"  Mean: {author_miss.mean():.2f}%")
    print(f"  Max:  {author_miss.max():.2f}% ({author_miss.idxmax()})")
    items_over_5 = (author_miss > 5).sum()
    items_over_40 = (author_miss > 40).sum()
    print(f"  Items >5% missing:  {items_over_5}")
    print(f"  Items >40% missing: {items_over_40}")
    
    print(f"\nFoil items missingness:")
    print(f"  Mean: {foil_miss.mean():.2f}%")
    print(f"  Max:  {foil_miss.max():.2f}% ({foil_miss.idxmax()})")
    
    # Endorsement rate summary (authors)
    endorse = sample_df[author_cols].mean()
    print(f"\nAuthor endorsement rates:")
    print(f"  Mean: {endorse.mean():.3f}")
    print(f"  SD:   {endorse.std():.3f}")
    print(f"  Min:  {endorse.min():.3f} ({endorse.idxmin()})")
    print(f"  Max:  {endorse.max():.3f} ({endorse.idxmax()})")
    
    # False alarm summary (foils)
    fa = sample_df[foil_cols].mean()
    print(f"\nFoil false alarm rates:")
    print(f"  Mean: {fa.mean():.3f}")
    print(f"  SD:   {fa.std():.3f}")
    print(f"  Max:  {fa.max():.3f} ({fa.idxmax()})")
    
    # Demographics summary
    print(f"\nDemographics:")
    print(f"  Age: M = {sample_df['age'].mean():.1f}, SD = {sample_df['age'].std():.1f}")
    print(f"  Sex: {sample_df['sex '].value_counts().to_dict()}")
    print(f"  Humanities: {sample_df['humanities or not'].value_counts().to_dict()}")

validate_sample(sample1, 'Sample 1 (Pretest)', author_cols, foil_cols)
validate_sample(sample2, 'Sample 2 (Validation)', author_cols, foil_cols)


Validation: Sample 1 (Pretest)  (N = 1035)
Missing author columns: 0
Missing foil columns:   0

Author items missingness:
  Mean: 0.56%
  Max:  56.52% (Yuri Tsypkin)
  Items >5% missing:  1
  Items >40% missing: 1

Foil items missingness:
  Mean: 0.00%
  Max:  0.00% (Gerrit HoogenbuM fill1)

Author endorsement rates:
  Mean: 0.513
  SD:   0.258
  Min:  0.043 (Yustein Gordier)
  Max:  0.981 (Jules Verne)

Foil false alarm rates:
  Mean: 0.044
  SD:   0.040
  Max:  0.219 (Paul Williams fill 37)

Demographics:
  Age: M = 27.5, SD = 9.7
  Sex: {'F': 692, 'M': 343}
  Humanities: {'-': 667, '+': 231, '_': 3}

Validation: Sample 2 (Validation)  (N = 800)
Missing author columns: 0
Missing foil columns:   0

Author items missingness:
  Mean: 0.00%
  Max:  0.12% (Ian Fleming)
  Items >5% missing:  0
  Items >40% missing: 0

Foil items missingness:
  Mean: 0.71%
  Max:  72.00% (Andrea Segre fill 93)

Author endorsement rates:
  Mean: 0.416
  SD:   0.307
  Min:  0.009 (Dmitry Bykov)
  Max:  0.984

## 7. Verify identical item sets across samples

In [8]:
# Both samples should have the exact same set of item columns
s1_items = set(item_cols)
s2_items = set(item_cols)  # same columns, different rows

assert s1_items == s2_items, "Item column sets differ between samples!"
print(f"Item column sets are identical: {len(s1_items)} items")
print(f"  Real authors: {len(author_cols)}")
print(f"  Foils:        {len(foil_cols)}")

Item column sets are identical: 204 items
  Real authors: 101
  Foils:        103


## 8. Export sample-level datasets and metadata

In [9]:
# Export full samples (demo + all items + source)
s1_path = os.path.join(OUT_DIR, "sample1_pretest.csv")
s2_path = os.path.join(OUT_DIR, "sample2_validation.csv")
meta_path = os.path.join(OUT_DIR, "item_metadata.csv")

sample1.to_csv(s1_path, index=False)
sample2.to_csv(s2_path, index=False)

# Copy item metadata with author/foil column lists
meta_out = meta.copy()
meta_out.to_csv(meta_path, index=False)

# Also save author-only response matrices (no demo, no foils)
s1_authors_path = os.path.join(OUT_DIR, "sample1_authors_only.csv")
s2_authors_path = os.path.join(OUT_DIR, "sample2_authors_only.csv")

sample1[author_cols].to_csv(s1_authors_path, index=False)
sample2[author_cols].to_csv(s2_authors_path, index=False)

print("Exported files:")
for p in [s1_path, s2_path, meta_path, s1_authors_path, s2_authors_path]:
    size = os.path.getsize(p)
    print(f"  {os.path.basename(p):40s} {size/1024:.1f} KB")

Exported files:
  sample1_pretest.csv                      483.2 KB
  sample2_validation.csv                   382.3 KB
  item_metadata.csv                        8.0 KB
  sample1_authors_only.csv                 210.0 KB
  sample2_authors_only.csv                 164.0 KB


## 9. Summary

In [10]:
print("="*60)
print("DATA PREPARATION COMPLETE")
print("="*60)
print(f"")
print(f"Sample 1 (Pretest):     N = {len(sample1):,}")
print(f"Sample 2 (Validation):  N = {len(sample2):,}")
print(f"Total items (cleaned):  {len(item_cols)}")
print(f"  Real authors:         {len(author_cols)}")
print(f"  Foils:                {len(foil_cols)}")
print(f"")
print(f"Output directory: {OUT_DIR}")
print(f"")
print("Files for downstream notebooks:")
print("  sample1_pretest.csv       — full sample 1 with demo+items+source")
print("  sample2_validation.csv    — full sample 2 with demo+items+source")
print("  sample1_authors_only.csv  — author response matrix, sample 1")
print("  sample2_authors_only.csv  — author response matrix, sample 2")
print("  item_metadata.csv         — item index, label, code, type")

DATA PREPARATION COMPLETE

Sample 1 (Pretest):     N = 1,035
Sample 2 (Validation):  N = 800
Total items (cleaned):  204
  Real authors:         101
  Foils:                103

Output directory: /home/polina/Documents/Cursor_Projects/Russian Author Recognition Test Cursor/data/processed/irt_pipeline

Files for downstream notebooks:
  sample1_pretest.csv       — full sample 1 with demo+items+source
  sample2_validation.csv    — full sample 2 with demo+items+source
  sample1_authors_only.csv  — author response matrix, sample 1
  sample2_authors_only.csv  — author response matrix, sample 2
  item_metadata.csv         — item index, label, code, type
